# 🤟 Wasel v4 Pro — Live Sign Language Translator
### MediaPipe Holistic + Self-Training + Real-Time Translation

**3 Steps:**
1. **Cell 1:** Install dependencies
2. **Cell 2:** Record your signs & train the AI (5 sec per sign)
3. **Cell 3:** Launch live translator with public URL!

> ⚡ **Zero external dependencies. The AI learns YOUR signs in seconds!**

In [ ]:
#@title 📦 Cell 1: Install & Setup

print("⏳ Installing dependencies...")
!pip install -q "gradio>=5.0.0" "mediapipe>=0.10.0" "scikit-learn>=1.3.0" "opencv-python-headless"

import cv2, pickle, time, os, logging
import numpy as np
import mediapipe as mp
from collections import deque
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("wasel")

# ─── MediaPipe Holistic ───
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_features(results):
    """Extract 225 features from MediaPipe Holistic results."""
    features = []
    # Pose: 33 landmarks × 3 (x,y,z) = 99
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 99)
    # Left hand: 21 × 3 = 63
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    # Right hand: 21 × 3 = 63
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y, lm.z])
    else:
        features.extend([0.0] * 63)
    return np.array(features)  # Total = 225

print("✅ MediaPipe Holistic loaded (pose 33 + hands 21×2 = 75 landmarks = 225 features)")
print("🎉 Ready! Proceed to Cell 2.")

In [ ]:
#@title 🎓 Cell 2: Record Signs & Train AI
#@markdown ### Instructions:
#@markdown 1. Type the sign name (e.g. "hello", "water", "help")
#@markdown 2. Click **Record** — perform the sign for 5 seconds
#@markdown 3. Repeat for each sign (minimum 3 signs)
#@markdown 4. Click **Train** when done!

import gradio as gr
import threading

# Storage
training_data = {}  # {label: [features_array, ...]}
classifier = None
label_encoder = None
is_recording = False
current_label = ""
record_buffer = []

def start_recording(label, image):
    """Process a webcam frame during recording."""
    global is_recording, current_label, record_buffer

    if not label or not label.strip():
        return image, "❌ Enter a sign name first!"

    label = label.strip().lower()
    current_label = label

    if image is None:
        return None, "❌ No camera image!"

    # Process frame
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = holistic.process(rgb)
    annotated = image.copy()

    # Draw skeleton
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(annotated, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(annotated, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(annotated, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

    # Extract and save features
    feat = extract_features(results)
    if label not in training_data:
        training_data[label] = []
    training_data[label].append(feat)

    # Status
    count = len(training_data[label])
    cv2.putText(annotated, f"RECORDING: {label} ({count} frames)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    status_lines = [f"📊 Recording '{label}': {count} frames captured"]
    status_lines.append(f"\n📋 All recorded signs:")
    for k, v in training_data.items():
        status_lines.append(f"   • {k}: {len(v)} frames")

    return annotated, "\n".join(status_lines)

def train_model():
    """Train classifier on recorded data."""
    global classifier, label_encoder

    if len(training_data) < 2:
        return "❌ Record at least 2 different signs first!"

    X, y = [], []
    for label, features_list in training_data.items():
        for feat in features_list:
            X.append(feat)
            y.append(label)

    X = np.array(X)
    y = np.array(y)

    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)

    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    if len(X) > 10:
        X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42)
        clf.fit(X_train, y_train)
        acc = accuracy_score(y_test, clf.predict(X_test))
    else:
        clf.fit(X, y_enc)
        acc = 1.0

    classifier = clf

    # Save model
    with open('/content/psl_classifier_self.pkl', 'wb') as f:
        pickle.dump((classifier, label_encoder), f)

    result = f"🎉 Training Complete!\n"
    result += f"   ✅ Accuracy: {acc*100:.1f}%\n"
    result += f"   ✅ Signs learned: {list(label_encoder.classes_)}\n"
    result += f"   ✅ Total samples: {len(X)}\n"
    result += f"   ✅ Features per sample: {X.shape[1]}\n\n"
    result += f"🚀 Now proceed to Cell 3 to launch live translator!"
    return result

def clear_data():
    """Clear all recorded data."""
    global training_data
    training_data = {}
    return "🗑️ All recorded data cleared. Start fresh!"

# ─── Gradio Recording Interface ───
with gr.Blocks(title="Wasel v4 — Sign Recorder") as recorder:
    gr.Markdown("## 🎓 Wasel v4 — Teach the AI Your Signs")
    gr.Markdown("Enter a sign name, show the sign to the webcam, and the AI records your pose + hand features.")

    with gr.Row():
        sign_name = gr.Textbox(label="Sign Name", placeholder="e.g. hello, water, help, pakistan...", scale=2)

    with gr.Row():
        webcam = gr.Image(sources=["webcam"], streaming=True, label="📸 Show your sign")
        output_img = gr.Image(label="🦴 Skeleton Preview")

    status = gr.Textbox(label="Status", lines=8, interactive=False)

    with gr.Row():
        train_btn = gr.Button("🧠 Train AI", variant="primary", scale=2)
        clear_btn = gr.Button("🗑️ Clear All", variant="stop", scale=1)

    # Events
    webcam.stream(fn=start_recording, inputs=[sign_name, webcam], outputs=[output_img, status])
    train_btn.click(fn=train_model, outputs=[status])
    clear_btn.click(fn=clear_data, outputs=[status])

print("🎉 Launching Sign Recorder...")
print("📋 A public URL will appear below.")
print("⚠️  Record at least 2-3 signs, then click Train!\n")
recorder.launch(share=True, quiet=False)

In [ ]:
#@title 🎥 Cell 3: Launch Live Translator
#@markdown Uses the model you just trained!

import gradio as gr
import pickle
import cv2
import numpy as np
from collections import deque

# ─── Load trained model ───
MODEL_PATH = '/content/psl_classifier_self.pkl'
if not os.path.exists(MODEL_PATH):
    raise RuntimeError("❌ No trained model found! Run Cell 2 first and click Train.")

with open(MODEL_PATH, 'rb') as f:
    classifier, label_encoder = pickle.load(f)

print(f"✅ Loaded model: {len(label_encoder.classes_)} signs")
print(f"🏷️ Signs: {list(label_encoder.classes_)}")

buf = deque(maxlen=10)

def translate_frame(image):
    """Live translation: detect pose + hands, predict sign."""
    if image is None:
        return image
    try:
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        out = image.copy()

        # Draw full skeleton (body + hands)
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS, mp_styles.get_default_pose_landmarks_style())
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, mp_styles.get_default_hand_landmarks_style())

        # Extract features & predict
        feat = extract_features(results)
        buf.append(feat)

        label_text = "Waiting for signs..."
        if len(buf) >= 3:
            avg = np.mean(np.array(list(buf)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 30.0:
                label = label_encoder.inverse_transform([idx])[0]
                label_text = f"{label.upper()} ({conf:.1f}%)"

        # Big green text
        cv2.putText(out, label_text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        return out
    except Exception as e:
        logger.error(f"Frame error: {e}")
        return image

# ─── Launch ───
demo = gr.Interface(
    fn=translate_frame,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Camera"),
    outputs=gr.Image(label="🤟 Live Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Live Sign Language Translator",
    description="The AI translates your signs in real-time! Show the signs you recorded in Cell 2.",
)

print("\n🎉 Launching Live Translator...")
print("📋 Share the public URL with anyone!")
demo.launch(share=True, quiet=False)